In [1]:
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
print("Dataset folder:", path)
print("Files:", os.listdir(path))

Dataset folder: /home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1
Files: ['test.csv', 'train.csv']


In [2]:
files = os.listdir(path)

if "train.csv" in files and "test.csv" in files:
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df  = pd.read_csv(os.path.join(path, "test.csv"))
    df = pd.concat([train_df, test_df], ignore_index=True)
else:
    csvs = [f for f in files if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csvs[0]))

df.head()

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [3]:
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})
df["satisfaction"].value_counts()

satisfaction
0    73452
1    56428
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib, os

X = df.drop(columns=["satisfaction"])
y = df["satisfaction"]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ]
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=200))
])

clf.fit(X_train, y_train)

def eval_set(name, Xs, ys):
    proba = clf.predict_proba(Xs)[:,1]
    pred = (proba >= 0.5).astype(int)
    return name, accuracy_score(ys, pred), f1_score(ys, pred), roc_auc_score(ys, proba)

print("VALID (acc, f1, auc):", eval_set("valid", X_valid, y_valid))
print("TEST  (acc, f1, auc):", eval_set("test", X_test, y_test))

VALID (acc, f1, auc): ('valid', 0.8734729493891797, 0.8514612835191323, 0.926597096570966)
TEST  (acc, f1, auc): ('test', 0.8739862437121445, 0.8517422549670873, 0.9276691200616554)


In [5]:
os.makedirs("../model", exist_ok=True)
joblib.dump(clf, "../model/model.pkl")
print("Saved to ../model/model.pkl")

Saved to ../model/model.pkl


In [6]:
import sagemaker, boto3, os
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction"

s3 = boto3.client("s3")

# upload model.pkl
local_model_path = os.path.abspath("../model/model.pkl")
s3_key = f"{prefix}/model/model.pkl"
s3.upload_file(local_model_path, bucket, s3_key)

print("Uploaded:", f"s3://{bucket}/{s3_key}")

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/model/model.pkl


In [7]:
import kagglehub, os, pandas as pd

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
files = os.listdir(path)

if "train.csv" in files and "test.csv" in files:
    train_df = pd.read_csv(os.path.join(path, "train.csv"))
    test_df  = pd.read_csv(os.path.join(path, "test.csv"))
    df = pd.concat([train_df, test_df], ignore_index=True)
else:
    csvs = [f for f in files if f.endswith(".csv")]
    df = pd.read_csv(os.path.join(path, csvs[0]))

# Minimal cleaning for training job input
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
df.to_csv("train_full.csv", index=False)

print("Saved:", os.path.abspath("train_full.csv"), "rows:", df.shape[0])

Saved: /mnt/custom-file-systems/s3/shared/train_full.csv rows: 129880


In [8]:
import sagemaker, boto3

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction/data"

s3_uri = sess.upload_data("train_full.csv", bucket=bucket, key_prefix=prefix)
print("Uploaded to:", s3_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Uploaded to: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/data/train_full.csv


In [14]:
import os

os.makedirs("src", exist_ok=True)

train_script = r'''
import argparse
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", type=str, default="/opt/ml/input/data/training")
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "."))
    args = parser.parse_args()

    files = [f for f in os.listdir(args.data_dir) if f.endswith(".csv")]
    if not files:
        raise FileNotFoundError(f"No CSV found in {args.data_dir}. Found: {os.listdir(args.data_dir)}")

    csv_path = os.path.join(args.data_dir, files[0])
    df = pd.read_csv(csv_path)

    df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")
    df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})

    if df["satisfaction"].isna().any():
        raise ValueError("Target mapping produced NaNs. Check satisfaction values.")

    X = df.drop(columns=["satisfaction"])
    y = df["satisfaction"]

    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler())]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
        ]
    )

    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=300))
    ])

    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    clf.fit(X_train, y_train)

    os.makedirs(args.model_dir, exist_ok=True)
    out_path = os.path.join(args.model_dir, "model.pkl")
    joblib.dump(clf, out_path)
    print("Saved model to:", out_path)

if __name__ == "__main__":
    main()
'''

with open("src/train_sagemaker.py", "w") as f:
    f.write(train_script)

print("Created:", os.path.abspath("src/train_sagemaker.py"))
print("src files:", os.listdir("src"))

Created: /mnt/custom-file-systems/s3/shared/src/train_sagemaker.py
src files: ['train_sagemaker.py']


In [15]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

role = sagemaker.get_execution_role()

estimator = SKLearn(
    entry_point="train_sagemaker.py",
    source_dir="src",
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.2-1",
    py_version="py3",
)

estimator.fit({"training": s3_uri})

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment
2026-02-27 05:35:53 Starting - Starting the training job...
2026-02-27 05:36:10 Starting - Preparing the instances for training...
2026-02-27 05:36:57 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-

In [16]:
print("Model artifact (S3):", estimator.model_data)

Model artifact (S3): s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/shared/sagemaker-scikit-learn-2026-02-27-05-35-50-152/output/model.tar.gz


In [17]:
import os, tarfile
import sagemaker

sess = sagemaker.Session()

local_tar = "model.tar.gz"
sess.download_data(path=".", bucket=estimator.model_data.split("/")[2],
                   key_prefix="/".join(estimator.model_data.split("/")[3:]))

# The download_data call saves into a folder sometimes; easiest approach:
print("Downloaded files:", os.listdir(".")) 

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Downloaded files: ['.ipynb_checkpoints', '.libs.json', '.temp_sagemaker_unified_studio_debugging_info', 'airline-satisfaction.AWS.ipynb', 'sagemaker-scikit-learn-2026-02-27-05-35-50-152', 'src', 'train_full.csv', 'model.tar.gz']


In [18]:
import tarfile, os
os.makedirs("model", exist_ok=True)

with tarfile.open("model.tar.gz", "r:gz") as tar:
    tar.extractall(path="model")

print("Extracted model folder contents:", os.listdir("model"))

Extracted model folder contents: ['model.pkl']


In [21]:
print("Sample shape:", X_sample.shape)
print("Columns in sample:", list(X_sample.columns)[:10])
print("Total columns:", len(X_sample.columns))

Sample shape: (1, 22)
Columns in sample: ['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location']
Total columns: 22


In [22]:
pipe.feature_names_in_

array(['Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class',
       'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding',
       'Seat comfort', 'Inflight entertainment', 'On-board service',
       'Leg room service', 'Baggage handling', 'Checkin service',
       'Inflight service', 'Cleanliness', 'Departure Delay in Minutes',
       'Arrival Delay in Minutes'], dtype=object)

In [3]:
import pandas as pd

df = pd.read_csv("train_full.csv")
df.head()

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,Female,Loyal Customer,25,Business travel,Business,562,2,5,5,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,Male,Loyal Customer,61,Business travel,Business,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [6]:
import sys
!{sys.executable} -m pip install -U "scikit-learn==1.2.2"

In [2]:
import sklearn, sys
print("sklearn:", sklearn.__version__)
print("python:", sys.executable)

sklearn: 1.2.2
python: /opt/conda/bin/python


In [5]:
import joblib
pipe = joblib.load("model/model.pkl")

state = pipe.__getstate__()
print("Pickle sklearn version:", state.get("_sklearn_version", "NOT FOUND"))

# Also check the OneHotEncoder object itself
pre = pipe.named_steps["preprocess"]
cat_pipe = pre.named_transformers_["cat"]
ohe = cat_pipe.named_steps["onehot"]

print("OHE class:", type(ohe))
print("Has _drop_idx_after_grouping?", hasattr(ohe, "_drop_idx_after_grouping"))

Pickle sklearn version: 1.2.2
OHE class: <class 'sklearn.preprocessing._encoders.OneHotEncoder'>
Has _drop_idx_after_grouping? False


In [6]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# 1) Load data
df = pd.read_csv("train_full.csv")
df = df.drop(columns=["Unnamed: 0", "id"], errors="ignore")

# 2) Target encoding
df["satisfaction"] = df["satisfaction"].map({"satisfied": 1, "neutral or dissatisfied": 0})

# 3) Split
X = df.drop(columns=["satisfaction"])
y = df["satisfaction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) Preprocess
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ]
)

# 5) Model
model = LogisticRegression(max_iter=500)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# 6) Fit
pipe.fit(X_train, y_train)

# 7) Evaluate
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

print("Accuracy:", round(acc, 4))
print("ROC AUC:", round(auc, 4))

# 8) Save
os.makedirs("model", exist_ok=True)
joblib.dump(pipe, "model/model.pkl")
print("Saved fresh model to model/model.pkl")

Accuracy: 0.8742
ROC AUC: 0.9273
Saved fresh model to model/model.pkl


In [7]:
import joblib, pandas as pd

df = pd.read_csv("train_full.csv")
pipe = joblib.load("model/model.pkl")

X_sample = df.drop(columns=["satisfaction"]).iloc[[0]].copy()
pred = pipe.predict(X_sample)[0]
proba = pipe.predict_proba(X_sample)[0, 1]

print("Prediction:", pred, "Prob(satisfied):", round(float(proba), 4))

Prediction: 0 Prob(satisfied): 0.2085


In [2]:
import os, glob
print("CWD:", os.getcwd())

# show top-level files/folders
print("Top level:", os.listdir("."))

# search for CSV anywhere under current folder
csvs = glob.glob("**/*.csv", recursive=True)
print("Found CSVs:", csvs[:50])  # show up to 50

CWD: /mnt/custom-file-systems/s3/shared
Top level: ['.ipynb_checkpoints', '.libs.json', '.temp_sagemaker_unified_studio_debugging_info', 'airline-satisfaction.AWS.ipynb', 'model.tar.gz', 'model', 'sagemaker-scikit-learn-2026-02-27-05-35-50-152', 'src', 'train_full.csv']
Found CSVs: ['train_full.csv']


In [5]:
!pip install kagglehub

  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.15-py3-none-any.whl.metadata (13 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.15-py3-none-any.whl (160 kB)


In [1]:
import kagglehub, os, glob

path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")

print("KaggleHub path:", path)
print("Files:", os.listdir(path))

csvs = glob.glob(os.path.join(path, "*.csv"))
print("CSV files:", csvs)

KaggleHub path: /home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1
Files: ['test.csv', 'train.csv']
CSV files: ['/home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1/test.csv', '/home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1/train.csv']


In [2]:
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()

prefix = "airline-satisfaction"

local_csv = csvs[0]   # pick the CSV you saw printed

sess.upload_data(local_csv, bucket=bucket, key_prefix=f"{prefix}/train")

train_s3_uri = f"s3://{bucket}/{prefix}/train/"
print("Train S3 URI:", train_s3_uri)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Train S3 URI: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/train/


In [3]:
import pandas as pd
df = pd.read_csv(local_csv)
print(df.columns)
print(df["satisfaction"].value_counts() if "satisfaction" in df.columns else "No 'satisfaction' column found")

Index(['Unnamed: 0', 'id', 'Gender', 'Customer Type', 'Age', 'Type of Travel',
       'Class', 'Flight Distance', 'Inflight wifi service',
       'Departure/Arrival time convenient', 'Ease of Online booking',
       'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
       'Inflight entertainment', 'On-board service', 'Leg room service',
       'Baggage handling', 'Checkin service', 'Inflight service',
       'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
       'satisfaction'],
      dtype='object')
satisfaction
neutral or dissatisfied    14573
satisfied                  11403
Name: count, dtype: int64


In [6]:
import os
os.makedirs("sm_job", exist_ok=True)

train_script = r'''
import os
import argparse
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
    parser.add_argument("--train-file", type=str, default="train.csv")  # OK if unknown; we'll auto-detect
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--random-state", type=int, default=42)
    return parser.parse_args()

def main():
    args = parse_args()

    data_path = os.path.join(args.train, args.train_file)

    # Auto-detect if filename doesn't match
    if not os.path.exists(data_path):
        files = [f for f in os.listdir(args.train) if f.endswith(".csv")]
        if len(files) == 0:
            raise FileNotFoundError(f"No CSV found in {args.train}. Files: {os.listdir(args.train)}")
        data_path = os.path.join(args.train, files[0])
        print("Auto-detected training file:", data_path)

    df = pd.read_csv(data_path)

    # Drop useless index column if present
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    if "satisfaction" not in df.columns:
        raise ValueError("Target column 'satisfaction' not found.")

    X = df.drop("satisfaction", axis=1)
    y = df["satisfaction"]

    # Separate columns
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]

    num_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop"
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        random_state=args.random_state,
        n_jobs=-1
    )

    model = Pipeline(steps=[
        ("preprocess", pre),
        ("clf", clf),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=args.test_size,
        random_state=args.random_state,
        stratify=y
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = float(accuracy_score(y_test, preds))
    print(f"[2b] Accuracy: {acc:.4f}")

    # CRITICAL: save to /opt/ml/model for endpoint hosting
    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    print("Saved model to SM_MODEL_DIR")

if __name__ == "__main__":
    main()
'''
with open("sm_job/train_sagemaker.py", "w") as f:
    f.write(train_script)

print("Created sm_job/train_sagemaker.py")

Created sm_job/train_sagemaker.py


In [7]:
!pip -q uninstall -y sagemaker
!pip -q install sagemaker==2.224.0
import sagemaker
print("SageMaker version:", sagemaker.__version__)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-ai 2.31.7 requires faiss-cpu!=1.8.0.post0,<2.0.0,>=1.8.0, which is not installed.
dask 2026.1.1 requires cloudpickle>=3.0.0, but you have cloudpickle 2.2.1 which is incompatible.
distributed 2026.1.1 requires cloudpickle>=3.0.0, but you have cloudpickle 2.2.1 which is incompatible.
grpcio-status 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
sagemaker-studio-analytics-extension 0.2.4 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which is incompatible.
SageMaker version: 2.254.1


In [8]:
import sagemaker
sess = sagemaker.Session()
bucket = sess.default_bucket()
print("Bucket:", bucket)

prefix = "airline-satisfaction"

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Bucket: amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3


In [9]:
import kagglehub
import os, glob
import sagemaker

# Download dataset
path = kagglehub.dataset_download("teejmahal20/airline-passenger-satisfaction")
print("Dataset path:", path)

# Find CSV file
csv_files = glob.glob(os.path.join(path, "*.csv"))
print("CSV files found:", csv_files)

# Use the first CSV
local_csv = csv_files[0]

# Upload to S3 (same method as Wine project)
sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "airline-satisfaction"

sess.upload_data(local_csv, bucket=bucket, key_prefix=f"{prefix}/train")

train_s3_uri = f"s3://{bucket}/{prefix}/train/"
print("Train S3 URI:", train_s3_uri)

Dataset path: /home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1
CSV files found: ['/home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1/test.csv', '/home/sagemaker-user/.cache/kagglehub/datasets/teejmahal20/airline-passenger-satisfaction/versions/1/train.csv']
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Train S3 URI: s3://amazon-sagemaker-913524921725-us-east-1-cp292gcmooo3l3/airline-satisfaction/train/


In [10]:
estimator.fit({"train": train_s3_uri})

2026-02-27 12:46:46 Starting - Starting the training job...
2026-02-27 12:47:00 Starting - Preparing the instances for training...
2026-02-27 12:47:48 Downloading - Downloading the training image......../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-27 12:48:56,217 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-02-27 12:48:56,220 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-02-27 12:48:56,223 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-02-27 12:48:56,239 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-02-27 12:48:56,468 s

In [14]:
import os
import json
import joblib
import pandas as pd

def model_fn(model_dir):
    # load what train_sagemaker.py saved
    model_path = os.path.join(model_dir, "model.joblib")
    return joblib.load(model_path)

def input_fn(request_body, request_content_type):
    # SageMaker ping uses this? Not always. But keep it safe.
    if request_content_type == "application/json":
        payload = json.loads(request_body)

        # allow {"instances":[{...},{...}]}
        if isinstance(payload, dict) and "instances" in payload:
            return pd.DataFrame(payload["instances"])

        # allow single dict
        if isinstance(payload, dict):
            return pd.DataFrame([payload])

    # fallback: allow CSV (optional)
    if request_content_type == "text/csv":
        from io import StringIO
        return pd.read_csv(StringIO(request_body), header=None)

    raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    preds = model.predict(input_data)
    return preds

def output_fn(prediction, response_content_type):
    return json.dumps({"prediction": prediction.tolist()}), "application/json"

In [15]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

endpoint_name = "airline-satisfaction-endpoint"

predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    entry_point="inference.py",
    source_dir="sm_job",
    endpoint_name=endpoint_name
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Endpoint created:", endpoint_name)

----------------------------------------------*

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3                                                                                             │
│    4 endpoint_name = "airline-satisfaction-endpoint"                                             │
│    5                                                                                             │
│ ❱  6 predictor = estimator.deploy(                                                               │
│    7 │   initial_instance_count=1,                                                               │
│    8 │   instance_type="ml.m5.large",                                                            │
│    9 │   entry_point="inference.py",                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/estimator.py:1771 in deploy                    │
│                                                                                                  │
│   1768 │   │   │   framework_version (str): Framework version of the Model Package Container Im  │
│   1769 │   │   │   │   (default: None).                                                          │
│   1770 │   │   │   nearest_model_name (str): Name of a pre-trained machine learning benchmarked  │
│ ❱ 1771 │   │   │   │   Amazon SageMaker Inference Recommender (default: None).                   │
│   1772 │   │   │   data_input_configuration (str): Input object for the model (default: None).   │
│   1773 │   │   │   skip_model_validation (str): Indicates if you want to skip model validation.  │
│   1774 │   │   │   │   Values can be "All" or "None" (default: None).                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/model.py:1814 in deploy                        │
│                                                                                                  │
│   1811 │   │   │   max_concurrent_transforms=max_concurrent_transforms,                          │
│   1812 │   │   │   max_payload=max_payload,                                                      │
│   1813 │   │   │   env=env,                                                                      │
│ ❱ 1814 │   │   │   tags=tags,                                                                    │
│   1815 │   │   │   base_transform_job_name=self._base_name or self.name,                         │
│   1816 │   │   │   volume_kms_key=volume_kms_key,                                                │
│   1817 │   │   │   sagemaker_session=self.sagemaker_session,                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/session.py:6250 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6247 │   │   Returns:                                                                          │
│   6248 │   │   │   Response dict from service.                                                   │
│   6249 │   │   """                                                                               │
│ ❱ 6250 │   │   search_args = {"Resource": resource}                                              │
│   6251 │   │                                                                                     │
│   6252 │   │   if search_expression:                       